# Pixels to Predictions — Inference Notebook

**Task:** SmolVLM-500M-Instruct + LoRA adapter + retrieval overlay. Loads a trained adapter and writes `submission.csv` for the Kaggle test set. Runs offline (no internet) once the model cache and adapter are available locally.

**AI tooling disclosure:** Claude Code (Opus) was used as a coding/debugging assistant during development of this project. All experimental design decisions, paper grounding, hyperparameter selections, and final analysis are documented in the accompanying report.

**Reproducibility:** all random seeds are fixed via `src.run._seed_all`. Default inference seed is `42`; training seed is `43` (matches `configs/train_lora_v3b_allmod_permute.yaml`).


In [ ]:
# Configure paths. Update to point at your local kagglehub cache + adapter.
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get('DATA_DIR', './data'))
ADAPTER_PATH = Path(os.environ.get('ADAPTER_PATH', './adapter_best'))
HF_CACHE = Path(os.environ.get('HF_HOME', '/scratch/$USER/deep-learning-final/hf_cache')).expanduser()
OUT_DIR = Path('./runs/inference')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_DIR={DATA_DIR}\nADAPTER_PATH={ADAPTER_PATH}\nHF_CACHE={HF_CACHE}\nOUT_DIR={OUT_DIR}')

In [ ]:
# Seed everything.
import sys; sys.path.insert(0, '..')
from src.run import _seed_all
_seed_all(42)

In [ ]:
# Configure offline mode and load model + adapter.
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'transformers')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
from peft import PeftModel
from transformers import AutoModelForVision2Seq, AutoProcessor

MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
processor = AutoProcessor.from_pretrained(MODEL_ID, local_files_only=True)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base = AutoModelForVision2Seq.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, local_files_only=True)
model = PeftModel.from_pretrained(base, str(ADAPTER_PATH), is_trainable=False)
if torch.cuda.is_available():
    model = model.to('cuda')
model.eval()
print('model + adapter loaded')

In [ ]:
# Run inference on the test split.
from src.data import load_split
from src.zero_shot import predict_zero_shot
from src.submission import write_submission

test_df = load_split(DATA_DIR / 'test.csv', labeled=False)
test_preds, _ = predict_zero_shot(
    test_df, data_dir=DATA_DIR, processor=processor, model=model,
    img_size=224, batch_size=48, num_workers=4, return_scores=True,
)
pred_map = dict(zip(test_df['id'], test_preds))
submission_path = write_submission(pred_map, test_df, OUT_DIR / 'submission.csv')
print(f'wrote {submission_path}')

## Optional: retrieval overlay (+2 pp on val)

Apply pHash + question-similarity + choice-match retrieval over training neighbors. Documented thresholds (`hamming_thresh=4`, `qsim_thresh=0.85`, `require_choice_match=True`) reproduce the +2 pp validation lift reported in the paper.


In [ ]:
!python ../scripts/retrieval_overlay.py \
    --base-submission {OUT_DIR}/submission.csv \
    --hamming-thresh 4 \
    --qsim-thresh 0.85 \
    --require-choice-match \
    --out {OUT_DIR}/retrieval/